# Self-Supervised Learning in NLP: BERT & GPT, Run Live

**Self-supervised learning** flips the usual recipe. Instead of paying humans to label data, we let *part of the input become the supervisory signal for predicting another part*. In Yann LeCun's framing, an input `x` is split into a piece `x'` that the model sees and a piece `x''` that it must predict — the data labels itself.

Modern language models are the headline success of this idea: **BERT** and **GPT** are *pre-trained on raw, unlabeled text* and only afterward adapted to real tasks. In this notebook we **see this live** by running ready-made Hugging Face pipelines on pretrained checkpoints — everything runs on a Colab CPU in a couple of minutes.

## The five-part arc

1. **Masked Language Modeling** — BERT masks tokens and predicts them; the most concrete instance of self-supervision.
2. **Why it works: contextualized embeddings** — the same token gets a *different* vector depending on context (fruit *apple* vs company *Apple*), shown as a cosine-similarity heatmap.
3. **Pre-train then fine-tune** — reuse one pre-trained encoder for downstream **sentiment classification** and **extractive QA** via off-the-shelf pipelines.
4. **The GPT branch** — autoregressive next-token prediction that powers open-ended text generation.
5. **In-context / few-shot learning** — GPT performing a task from a prompt with a few examples and **no gradient descent**.

## Demo philosophy

We do **not** pre-train from scratch — full pre-training needs billions of words and days of TPU time (the lecture's "training BERT is challenging" caveat). Instead we run **inference on pretrained checkpoints** plus interactive widgets, so every claim — masked-token prediction, word-sense separation, sentiment/QA, generation, and prompting — is something **you run and see**, not just read.

In [ ]:
# === Environment setup ===
# Colab has torch, numpy, matplotlib, and ipywidgets preinstalled; only transformers needs installing.
!pip install -q transformers
# !pip install -q ipywidgets   # fallback if ipywidgets is missing

import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import time
import transformers
from transformers import pipeline, AutoTokenizer, AutoModel, set_seed

# --- Reproducibility: fix all RNG seeds so Run All is deterministic ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Device selection (HF pipeline takes an int flag: 0=GPU, -1=CPU; AutoModel.to takes a string) ---
device = 0 if torch.cuda.is_available() else -1
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
# Checkpoints download from the Hugging Face Hub on first use, then are cached.

# --- Shared plotting defaults (used by every later bar chart / heatmap) ---
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['font.size'] = 10
BAR_COLOR = '#4C72B0'
HEATMAP_CMAP = 'viridis'

print(f'transformers {transformers.__version__} | device={device_str} | seed={SEED}')

In [ ]:
# === Shared toolkit: cached model loader + plotting helpers (defined once) ===
MODEL_CACHE = {}

# Lazily build and cache a Hugging Face pipeline so each checkpoint loads at most once.
def get_pipeline(task, model_name):
    key = (task, model_name)
    if key not in MODEL_CACHE:
        print('loading ' + task + ':' + model_name + ' ...')
        # Insert only AFTER successful construction so a Hub failure does not poison the cache.
        MODEL_CACHE[key] = pipeline(task, model=model_name, device=device)
    return MODEL_CACHE[key]

# Horizontal bar chart of token/label probabilities on the shared style.
def plot_token_probs(labels, probs, title='', ax=None):
    assert len(labels) == len(probs), 'labels and probs must have equal length'
    if len(labels) == 0:
        print('no predictions to plot')
        return None
    probs = [float(p) for p in probs]
    if ax is None:
        fig, ax = plt.subplots()
    y = list(range(len(labels)))
    ax.barh(y, probs, color=BAR_COLOR)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()                                  # highest probability on top
    ax.set_xlabel('probability')
    upper = max(probs) * 1.15 if max(probs) > 0 else 1e-3
    ax.set_xlim(0, upper)
    for i, p in enumerate(probs):
        ax.text(p + upper * 0.01, i, format(p, '.3f'), va='center', fontsize=9)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    return ax

# Pretty-print a ranked list of (token, probability).
def show_predictions(pairs):
    for rank, (token, prob) in enumerate(pairs, start=1):
        prob_str = format(prob, '.4f') if prob is not None else 'n/a'
        print(str(rank).rjust(2) + '. ' + str(token).ljust(15) + ' ' + prob_str)

# --- Sanity check: load the fill-mask pipeline once and confirm it ranks tokens ---
fm = get_pipeline('fill-mask', 'bert-base-uncased')
_sanity = fm('The capital of France is ' + fm.tokenizer.mask_token + '.')
assert isinstance(_sanity, list) and 'token_str' in _sanity[0] and 'score' in _sanity[0]
_tok = _sanity[0]['token_str'].strip()
_sc = _sanity[0]['score']
print('device=' + device_str + ' | top filling: ' + _tok + ' (' + format(_sc, '.3f') + ')')
print('cached models:', list(MODEL_CACHE.keys()))

## 1 · Masked Language Modeling (MLM)

Masked Language Modeling is the concrete realization of self-supervision. The recipe:

1. Randomly **mask ~15% of the tokens** in a sentence (replace each with a special `[MASK]` token — or occasionally a random word).
2. Feed the corrupted sentence through a Transformer **encoder**.
3. A **linear + softmax** head predicts the **original** token at each masked position.
4. Train by minimizing **cross-entropy** against the ground-truth token.

The key point: there are **no human labels**. The text supplies its own targets — `x''` = the masked token, `x'` = the surrounding context — exactly LeCun's split-the-input idea from the intro.

Because the encoder attends to **both left and right context**, BERT learns deep **bidirectional** representations of meaning.

> Next: a Hugging Face `fill-mask` pipeline lets us mask a word in a real sentence and watch BERT rank candidate fillings by probability — the very mechanism that, run over billions of sentences, **is** the pre-training objective.

In [ ]:
# === Concept 1: Masked Language Modeling -- static top-k prediction ===
fill_mask = get_pipeline('fill-mask', 'bert-base-uncased')
MASK = fill_mask.tokenizer.mask_token   # use the tokenizer's mask token, never hardcode '[MASK]'
TOP_K = 8

main_sentence = 'The capital of France is ' + MASK + '.'
context_sentence = 'I poured myself a cup of ' + MASK + ' in the morning.'

results = fill_mask(main_sentence, top_k=TOP_K)
labels = [r['token_str'].strip() for r in results]
probs = [r['score'] for r in results]

ax = plot_token_probs(labels, probs, title='BERT top-k fillings: ' + main_sentence)
mlm_prediction_figure = ax.figure
plt.show()

print('Ranked fillings for:', main_sentence)
show_predictions(list(zip(labels, probs)))

ctx_results = fill_mask(context_sentence, top_k=TOP_K)
print('Top filling for:', context_sentence)
print('  ->', ctx_results[0]['token_str'].strip(), '(' + format(ctx_results[0]['score'], '.3f') + ')')

# This mechanism -- predict the masked token from context -- applied over billions of unlabeled
# sentences IS BERT's self-supervised pre-training objective. fill_mask is reused by C06.

In [ ]:
# === Concept 1: interactive masked-language-model explorer ===
txt = widgets.Textarea(value='The [MASK] sat on the mat.', description='Sentence:',
                       layout=widgets.Layout(width='90%'))
k_slider = widgets.IntSlider(value=8, min=1, max=15, description='top-k')
run_btn = widgets.Button(description='Predict', button_style='primary')
out = widgets.Output()

def run_mlm(_):
    out.clear_output(wait=True)
    with out:
        sentence = txt.value
        k = k_slider.value
        if '[MASK]' not in sentence.upper():
            print('Please include the literal token [MASK] in your sentence.')
            return
        # Replace the user-typed [MASK] (any case) with the tokenizer's mask token.
        idx = sentence.lower().find('[mask]')
        masked = sentence[:idx] + fill_mask.tokenizer.mask_token + sentence[idx + 6:]
        try:
            results = fill_mask(masked, top_k=k)
        except Exception as e:
            print('Error during prediction:', e)
            return
        labels = [r['token_str'].strip() for r in results]
        probs = [r['score'] for r in results]
        plot_token_probs(labels, probs, title='Top-k fillings')
        plt.show()
        show_predictions(list(zip(labels, probs)))

run_btn.on_click(run_mlm)
mlm_widget = widgets.VBox([txt, k_slider, run_btn, out])
display(mlm_widget)
run_mlm(None)  # render a useful static default on first load

## 2 · Why BERT Works: Contextualized Word Embeddings

> "You shall know a word by the company it keeps." — J. R. Firth

A **static** word embedding (word2vec / CBOW) maps *every* occurrence of a word to the **same** vector. BERT does something different: it produces a **different vector for the same token depending on its sentence**, because self-attention mixes in the surrounding context.

So the character string *apple* in fruit sentences and *Apple* in company sentences end up with embeddings that cluster by **sense**, not by spelling.

> Next: we extract BERT's hidden-state vector for a target word across many sentences and render the pairwise **cosine-similarity heatmap**. Same-sense sentences light up as bright blocks while cross-sense pairs stay dark — reproducing the lecture's apple/Apple result and showing, visually, that the representation depends on context.

In [ ]:
# === Concept 2: contextual-embedding toolkit + static apple/Apple heatmap ===
_BERT_TOK = AutoTokenizer.from_pretrained('bert-base-uncased')
_BERT_ENC = AutoModel.from_pretrained('bert-base-uncased').to(device_str).eval()

# Find the contiguous span of input_tokens matching word_subtokens (case-insensitive, '##'-aware).
def _locate_word_span(input_tokens, word_subtokens):
    clean = [t.lstrip('#').lower() for t in input_tokens]
    target = [t.lstrip('#').lower() for t in word_subtokens]
    n = len(target)
    if n == 0:
        return []
    for i in range(len(clean) - n + 1):
        if clean[i:i + n] == target:
            return list(range(i, i + n))
    return []

# Return one contextual vector per sentence for `word` (shape (n_sentences, hidden)).
def bert_contextual_embedding(word, sentences):
    word_subtokens = _BERT_TOK.tokenize(word)
    vectors = []
    for sent in sentences:
        inputs = _BERT_TOK(sent, return_tensors='pt').to(device_str)
        with torch.no_grad():
            enc_out = _BERT_ENC(**inputs)
        hidden = enc_out.last_hidden_state[0]                  # (seq_len, hidden)
        tokens = _BERT_TOK.convert_ids_to_tokens(inputs['input_ids'][0])
        span = _locate_word_span(tokens, word_subtokens)
        if span:
            vec = hidden[span].mean(dim=0)
        else:
            print('warning: ' + repr(word) + ' not found in ' + repr(sent) + ' -- using [CLS] vector')
            vec = hidden[0]
        vec = vec.cpu().numpy()
        vec = vec / (np.linalg.norm(vec) + 1e-8)              # L2-normalize
        vectors.append(vec)
    return np.stack(vectors)

# Pairwise cosine-similarity heatmap over row-normalized `vectors`.
def cosine_heatmap(vectors, labels, ax=None):
    norm = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-8)
    sim = norm @ norm.T
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(sim, cmap=HEATMAP_CMAP, vmin=-1, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            txt_color = 'white' if sim[i, j] < 0.5 else 'black'
            ax.text(j, i, format(sim[i, j], '.2f'), ha='center', va='center', color=txt_color, fontsize=8)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title('Pairwise cosine similarity')
    plt.tight_layout()
    return ax

fruit_sentences = [
    'I ate a juicy apple for breakfast.',
    'She picked a ripe apple from the tree.',
    'An apple a day keeps the doctor away.',
    'The apple pie smelled delicious.',
    'He sliced a green apple into the salad.',
]
company_sentences = [
    'Apple released a new iPhone this year.',
    'Apple stock rose after the earnings call.',
    'She works as an engineer at Apple.',
    'Apple unveiled its latest MacBook laptop.',
    'The Apple logo is recognized worldwide.',
]
sentences = fruit_sentences + company_sentences
labels = (['F' + str(i + 1) for i in range(len(fruit_sentences))] +
          ['C' + str(i + 1) for i in range(len(company_sentences))])

vectors = bert_contextual_embedding('apple', sentences)
ax = cosine_heatmap(vectors, labels)
embedding_heatmap_figure = ax.figure
plt.show()

# Quantify the separation: within-sense vs across-sense mean cosine similarity.
norm = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-8)
sim = norm @ norm.T
nf = len(fruit_sentences)

def _block_mean(sub):
    m = sub.shape[0]
    if m < 2:
        return float('nan')
    return (sub.sum() - np.trace(sub)) / (m * m - m)

within = np.nanmean([_block_mean(sim[:nf, :nf]), _block_mean(sim[nf:, nf:])])
across = sim[:nf, nf:].mean()
print('mean within-sense cosine:  ' + format(within, '.3f'))
print('mean across-sense cosine:  ' + format(across, '.3f'))
print('A static word2vec embedding could NOT produce this split (one vector per word).')

In [ ]:
# === Concept 2: interactive contextual-embedding explorer ===
DEFAULT_SENTENCES = '\n'.join([
    'I ate a juicy apple for breakfast.',
    'She picked a ripe apple from the tree.',
    'An apple a day keeps the doctor away.',
    'Apple released a new iPhone this year.',
    'Apple stock rose after the earnings call.',
    'She works as an engineer at Apple.',
])

word_box = widgets.Text(value='apple', description='Word:')
sent_area = widgets.Textarea(value=DEFAULT_SENTENCES, description='Sentences (one per line):',
                             layout=widgets.Layout(width='90%', height='140px'))
run_btn = widgets.Button(description='Build heatmap', button_style='primary')
out = widgets.Output()

def run_embed(_):
    out.clear_output(wait=True)
    with out:
        sentences = [s for s in sent_area.value.splitlines() if s.strip()]
        word = word_box.value.strip()
        if len(sentences) < 2:
            print('Enter at least two sentences (one per line).')
            return
        if not word:
            print('Please enter a target word.')
            return
        try:
            vecs = bert_contextual_embedding(word, sentences)
        except Exception as e:
            print('Error computing embeddings:', e)
            return
        sent_labels = ['S' + str(i + 1) for i in range(len(sentences))]
        cosine_heatmap(vecs, sent_labels)
        plt.show()

run_btn.on_click(run_embed)
embeddings_widget = widgets.VBox([word_box, sent_area, run_btn, out])
display(embeddings_widget)
run_embed(None)  # static default heatmap on first load

## 3 · Pre-train, then Fine-tune: Downstream Tasks

The paradigm that made BERT useful:

- **Pre-train once** on huge unlabeled text (the expensive part — already done for us).
- **Attach a small task head** and **fine-tune** on a little labeled data per task.

Fine-tuning **converges faster** and **generalizes better** than training the same network from random initialization (the lecture's GLUE results and pre-train-vs-scratch loss curves).

### The four canonical BERT recipes

1. **Sequence classification** — `[CLS]` + linear head (e.g. sentiment).
2. **Sequence labeling** — a per-token linear head (POS tagging / NER).
3. **Two-sequence classification** — a sentence pair in, one label out (e.g. NLI).
4. **Extractive QA** — predict an answer **span** by pointing at **start** and **end** token positions via inner products + softmax over the document.

> Next: we run the two most demoable cases as ready-made **fine-tuned** pipelines — **sentiment classification** and **extractive question answering** — both reusing the same pre-trained encoder, just with different heads.

In [ ]:
# === Concept 3: downstream tasks -- sentiment classification + extractive QA ===
sentiment_pipeline = get_pipeline('sentiment-analysis', 'distilbert-base-uncased-finetuned-sst-2-english')
qa_pipeline = get_pipeline('question-answering', 'distilbert-base-cased-distilled-squad')

# --- Case 1: sequence classification (sentiment via [CLS] + linear head) ---
sentiment_examples = [
    'This movie was absolutely fantastic and moving.',
    'The food was cold and the service was terrible.',
    'It was fine, nothing special but not bad either.',
]
sent_results = sentiment_pipeline(sentiment_examples)
if isinstance(sent_results, dict):
    sent_results = [sent_results]

fig, ax = plt.subplots()
colors = ['#2ca02c' if r['label'].upper() == 'POSITIVE' else '#d62728' for r in sent_results]
ax.bar(range(len(sent_results)), [r['score'] for r in sent_results], color=colors)
ax.set_xticks(range(len(sent_results)))
ax.set_xticklabels(['#' + str(i + 1) + ' ' + r['label'] for i, r in enumerate(sent_results)])
ax.set_ylabel('confidence')
ax.set_ylim(0, 1.05)
ax.set_title('Sentiment confidence (green=POSITIVE, red=NEGATIVE)')
plt.tight_layout()
downstream_figure = fig
plt.show()

for s, r in zip(sentiment_examples, sent_results):
    lab, sc = r['label'], r['score']
    print('[' + lab + ' ' + format(sc, '.3f') + '] ' + s)

# --- Case 4: extractive QA (predict the answer SPAN via start/end pointers) ---
qa_context = (
    'The research team at the university built a small robot that can sort '
    'recyclable materials. It was funded by a government grant in 2021 and '
    'uses a camera and a neural network to identify plastic, glass, and metal.'
)
qa_questions = [
    'What did the team build?',
    'When was the project funded?',
    'What does the robot use to identify materials?',
]
print('Extractive QA:')
for q in qa_questions:
    res = qa_pipeline(question=q, context=qa_context)
    ans = res['answer']
    if not ans:
        print('Q: ' + q + ' -> no answer span found')
        continue
    span_ok = qa_context[res['start']:res['end']] == ans
    print('Q: ' + q)
    print('   -> ' + repr(ans) + ' (score=' + format(res['score'], '.3f') +
          ', start=' + str(res['start']) + ', end=' + str(res['end']) +
          ', span_ok=' + str(span_ok) + ')')

# NOTE: BOTH pipelines are fine-tuned members of the BERT family -- they reuse the SAME
# self-supervised pre-trained encoder; only the task-specific head differs.
# sentiment_pipeline and qa_pipeline are reused by C12.

In [ ]:
# === Concept 3: interactive downstream-task explorer ===
DEFAULT_CONTEXT = (
    'The research team at the university built a small robot that can sort '
    'recyclable materials. It was funded by a government grant in 2021.'
)

task_dd = widgets.Dropdown(options=['sentiment', 'question-answering'], value='sentiment', description='Task:')
sent_in = widgets.Text(value='I really enjoyed this movie.', description='Sentence:',
                       layout=widgets.Layout(width='90%'))
ctx_in = widgets.Textarea(value=DEFAULT_CONTEXT, description='Context:',
                          layout=widgets.Layout(width='90%', height='100px'))
q_in = widgets.Text(value='What did the team build?', description='Question:',
                    layout=widgets.Layout(width='90%'))
run_btn = widgets.Button(description='Run', button_style='primary')
out = widgets.Output()

def on_task_change(change=None):
    is_sent = task_dd.value == 'sentiment'
    sent_in.layout.display = '' if is_sent else 'none'
    ctx_in.layout.display = 'none' if is_sent else ''
    q_in.layout.display = 'none' if is_sent else ''

task_dd.observe(on_task_change, names='value')
on_task_change()

def run_task(_):
    out.clear_output(wait=True)
    with out:
        try:
            if task_dd.value == 'sentiment':
                text = sent_in.value.strip()
                if not text:
                    print('Please enter a sentence.')
                    return
                r = sentiment_pipeline(text)
                r = r[0] if isinstance(r, list) else r
                print('Sentiment: ' + r['label'] + '  (confidence ' + format(r['score'], '.3f') + ')')
                fig, ax = plt.subplots(figsize=(4, 1.2))
                color = '#2ca02c' if r['label'].upper() == 'POSITIVE' else '#d62728'
                ax.barh([0], [r['score']], color=color)
                ax.set_xlim(0, 1)
                ax.set_yticks([])
                ax.set_title(r['label'])
                plt.tight_layout()
                plt.show()
            else:
                context = ctx_in.value.strip()
                question = q_in.value.strip()
                if not context or not question:
                    print('Please enter both a context and a question.')
                    return
                res = qa_pipeline(question=question, context=context)
                print('Answer: ' + res['answer'])
                print('score=' + format(res['score'], '.3f') +
                      ', start=' + str(res['start']) + ', end=' + str(res['end']))
        except Exception as e:
            print('Error:', e)

run_btn.on_click(run_task)
downstream_widget = widgets.VBox([task_dd, sent_in, ctx_in, q_in, run_btn, out])
display(downstream_widget)
run_task(None)  # static default (sentiment) result on first load

## 4 · The GPT Branch: Autoregressive Next-Token Prediction

Where BERT is an **encoder** that looks both ways and fills in blanks, GPT is a Transformer **decoder** with **causal masking**: at each position it sees only the **past** tokens and predicts the **next** one.

Given `<BOS> 台 灣 大`, it predicts `學` — via a **linear + softmax** head trained by **cross-entropy**, with future positions masked out so the model cannot peek ahead.

This single objective, repeated as "predict the next token, append it, repeat," generates open-ended text and is the basis of modern large language models. GPT-2 can already write surprisingly coherent paragraphs. Note this is still **self-supervised** — the next token is its own label.

> Next: a GPT-2 generation pipeline produces a continuation, **and** we peek at the model's next-token probability distribution for a prompt to see generation as repeated sampling.

In [ ]:
# === Concept 4: GPT-2 autoregressive generation + next-token distribution ===
text_generator = get_pipeline('text-generation', 'gpt2')

prompt = 'In the future, artificial intelligence will'
gen = text_generator(prompt, max_new_tokens=40, do_sample=True, temperature=0.8,
                     top_k=50, pad_token_id=text_generator.tokenizer.eos_token_id)
generation_example = gen[0]['generated_text']
print('Prompt + generated continuation:')
print(generation_example)

# --- Expose the single autoregressive STEP underneath generation ---
tok = text_generator.tokenizer
model = text_generator.model
TOP_K = 10
inputs = tok(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits
next_logits = logits[0, -1, :]              # logits for the NEXT token
probs = torch.softmax(next_logits, dim=-1)
top = torch.topk(probs, TOP_K)
cand_labels = [tok.decode([i]).strip() for i in top.indices.tolist()]
cand_probs = top.values.tolist()

ax = plot_token_probs(cand_labels, cand_probs,
                      title='GPT-2 next-token distribution after: ' + prompt)
next_token_distribution_figure = ax.figure
plt.show()

# Generation = repeatedly draw from this distribution, append, and repeat.
# Greedy/argmax decoding always takes the top bar; sampling (C15) picks proportional to bar height.
# text_generator is reused by C15 and C17.

In [ ]:
# === Concept 4: interactive GPT-2 generation explorer ===
prompt_area = widgets.Textarea(value='Once upon a time,', description='Prompt:',
                               layout=widgets.Layout(width='90%', height='80px'))
len_slider = widgets.IntSlider(value=40, min=10, max=80, step=5, description='max tokens')
temp_slider = widgets.FloatSlider(value=0.8, min=0.1, max=1.5, step=0.1, description='temperature')
topk_slider = widgets.IntSlider(value=50, min=1, max=100, step=1, description='top-k')
topp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description='top-p')
run_btn = widgets.Button(description='Generate', button_style='primary')
out = widgets.Output()

def run_gen(_):
    out.clear_output(wait=True)
    with out:
        prompt = prompt_area.value.strip() or 'Once upon a time,'
        temperature = max(0.1, temp_slider.value)  # temperature=0 with sampling is invalid
        try:
            result = text_generator(
                prompt,
                max_new_tokens=len_slider.value,   # bounded by the slider max (80) so the kernel cannot hang
                do_sample=True,
                temperature=temperature,
                top_k=topk_slider.value,
                top_p=topp_slider.value,
                num_return_sequences=1,
                pad_token_id=text_generator.tokenizer.eos_token_id,
            )
            print(result[0]['generated_text'])
        except Exception as e:
            print('Error during generation:', e)

run_btn.on_click(run_gen)
generation_widget = widgets.VBox([prompt_area, len_slider, temp_slider, topk_slider, topp_slider, run_btn, out])
display(generation_widget)
run_gen(None)  # static default generation on first load

## 5 · In-Context (Few-Shot) Learning

GPT-3 showed that a sufficiently large autoregressive model can perform a **new task purely from its prompt** — a task description, then a few input→output demonstration examples, then the query — inferring the pattern **in context** with **no gradient descent and no weight update** (contrast this with the fine-tuning of Section 3).

The lecture's terminology:

- **Zero-shot** — description only.
- **One-shot** — one example.
- **Few-shot** — a handful of examples.

Aggregate accuracy over dozens of tasks **rises with model size** for all three settings — which is why this is a paradigm shift toward **prompting** rather than fine-tuning.

> Next: we build a few-shot prompt for a simple task and compare GPT-2's zero-shot vs few-shot completions — with the honest caveat that GPT-2 is small, so the effect is modest. It is **scale** (GPT-3 and beyond) that makes few-shot learning powerful.

In [ ]:
# === Concept 5: in-context learning -- static zero-shot vs few-shot ===
# Assemble a task description + (input, output) demos + final query into one prompt.
def build_prompt(task_description, examples, query):
    prompt = task_description.strip() + '\n'
    for inp, out_val in examples:
        prompt += 'Input: ' + inp + '\nOutput: ' + out_val + '\n'
    prompt += 'Input: ' + query + '\nOutput:'
    return prompt

# Slice off the prompt and keep only the first generated line.
def extract_completion(generated_text, prompt):
    tail = generated_text[len(prompt):]
    return tail.split('\n')[0].strip()

task_description = 'Translate the English word to French.'
examples = [('cat', 'chat'), ('dog', 'chien'), ('house', 'maison')]
query = 'book'

zero_shot = build_prompt(task_description, [], query)
few_shot = build_prompt(task_description, examples, query)

def _complete(prompt):
    gen = text_generator(prompt, max_new_tokens=10, do_sample=False,
                         pad_token_id=text_generator.tokenizer.eos_token_id)
    return extract_completion(gen[0]['generated_text'], prompt)

zero_out = _complete(zero_shot)
few_out = _complete(few_shot)

fewshot_comparison_figure = None  # this demo is textual; no figure produced
print('=== ZERO-SHOT PROMPT ===')
print(zero_shot)
print('  completion ->', repr(zero_out))
print()
print('=== FEW-SHOT PROMPT ===')
print(few_shot)
print('  completion ->', repr(few_out))

# CAVEAT: GPT-2 is small, so the few-shot effect is modest and sometimes imperfect.
# The lecture's point is that few-shot accuracy RISES with model SCALE (GPT-3 and beyond),
# which a classroom notebook cannot run. build_prompt is reused by C18.

In [ ]:
# === Concept 5: interactive in-context-learning explorer ===
desc_area = widgets.Textarea(value='Translate the English word to French.',
                             description='Task:', layout=widgets.Layout(width='90%', height='60px'))
ex_area = widgets.Textarea(value='cat => chat\ndog => chien\nhouse => maison',
                           description='Examples:', layout=widgets.Layout(width='90%', height='100px'))
query_box = widgets.Text(value='book', description='Query:')
shots_slider = widgets.IntSlider(value=3, min=0, max=5, description='shots')
run_btn = widgets.Button(description='Run', button_style='primary')
out = widgets.Output()

# Parse 'input => output' lines into (input, output) tuples, skipping malformed lines.
def parse_examples(text):
    pairs = []
    for line in text.splitlines():
        if '=>' not in line:
            continue
        left, right = line.split('=>', 1)
        if left.strip() and right.strip():
            pairs.append((left.strip(), right.strip()))
    return pairs

def run_icl(_):
    out.clear_output(wait=True)
    with out:
        query = query_box.value.strip()
        if not query:
            print('Please enter a query.')
            return
        examples = parse_examples(ex_area.value)
        n = min(shots_slider.value, len(examples))  # clamp shots to the available examples
        prompt = build_prompt(desc_area.value, examples[:n], query)
        print('--- Prompt (' + str(n) + ' shot' + ('' if n == 1 else 's') + ') ---')
        print(prompt)
        try:
            gen = text_generator(prompt, max_new_tokens=10, do_sample=False,
                                 pad_token_id=text_generator.tokenizer.eos_token_id)
            print('--- Completion ---')
            print(repr(extract_completion(gen[0]['generated_text'], prompt)))
        except Exception as e:
            print('Error during generation:', e)

run_btn.on_click(run_icl)
incontext_widget = widgets.VBox([desc_area, ex_area, query_box, shots_slider, run_btn, out])
display(incontext_widget)
run_icl(None)  # static default on first load

## Recap & Where to Go Next

We followed one spine, and **ran every claim** on pretrained checkpoints rather than asserting it:

1. **Masked Language Modeling** is the concrete self-supervised objective — predict masked tokens from bidirectional context.
2. **Contextualized word embeddings** explain *why* BERT works — the same token gets different vectors by context, visualized as a sense-separating cosine heatmap.
3. **Pre-train then fine-tune** turns the pre-trained encoder into downstream **sentiment** and **extractive-QA** systems.
4. **GPT's autoregressive next-token prediction** is the second branch of self-supervision and powers open-ended generation.
5. **In-context / few-shot learning** performs tasks from the prompt alone, with no gradient descent.

### Intentionally left out (candidates for a follow-up notebook)

To keep a classroom Run All fast and reliable, we skipped several high-value methods:

- **Next Sentence Prediction / Sentence Order Prediction** — RoBERTa shows NSP is unhelpful; ALBERT replaces it with SOP.
- **Seq2seq pre-training** (MASS / BART / T5) for summarization and translation.
- **Multilingual BERT** and cross-lingual zero-shot transfer.
- **Beyond text** — contrastive self-supervised vision (SimCLR / BYOL) and speech (wav2vec / SUPERB).

Each needs heavier models, data, or training than a classroom notebook allows — but every one builds on the same idea you just ran live: **let the data supervise itself.**